### INITIALIZATION: 
IMPORT TOOLS, SET SEED, AND CREATE INDEPENDENT TABLES

In [1]:
# libraries
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# seed
SEED = 42
np.random.seed(SEED)


print("Libraries loaded and seed set.")

Libraries loaded and seed set.


### Independent Entities

In [2]:
# Step 2: Independent Entities (Departments)
departments_data = [
    {"name": "Traffic & Transport", "daily_capacity": 50, "vulnerability_to_storm": 5.0, "base_rate": 20},
    {"name": "Public Works", "daily_capacity": 40, "vulnerability_to_storm": 8.0, "base_rate": 15},
    {"name": "Parks & Recreation", "daily_capacity": 10, "vulnerability_to_storm": 1.5, "base_rate": 2},
    {"name": "Animal Control", "daily_capacity": 15, "vulnerability_to_storm": 1.0, "base_rate": 5},
    {"name": "Sanitation", "daily_capacity": 60, "vulnerability_to_storm": 3.0, "base_rate": 45}
]

df_departments = pd.DataFrame(departments_data)

# Give each department a unique UUID
df_departments['department_id'] = [str(uuid.uuid4()) for _ in range(len(df_departments))]

print(df_departments.head())

                  name  ...                         department_id
0  Traffic & Transport  ...  77422a7c-b2e5-4ddd-a848-69c95deef1c2
1         Public Works  ...  d27083fb-c56c-4c68-bdfc-77181bcaadfe
2   Parks & Recreation  ...  fa86aaed-ebb5-4594-b1d2-2670908475ae
3       Animal Control  ...  f6996383-b11a-41e6-9743-a611b75893fb
4           Sanitation  ...  e9c61c7a-428d-44e3-81b9-929282f8e9e7

[5 rows x 5 columns]


In [3]:
# setup 
NUM_CITIZENS = 10000
BASE_DATE = datetime(2026, 8, 1)

# citizen ids
# generate 10000 ids for each citizen
citizen_ids = [str(uuid.uuid4()) for _ in range(NUM_CITIZENS)]

# dates
# generate random dates
random_days_ago = np.random.randint(1, 365,size=NUM_CITIZENS)
join_dates = [
    (BASE_DATE - timedelta(days=int(days))).strftime("%Y-%m-%d") 
    for days in random_days_ago
]

# clip L_civic
# np.random.normal creates the raw data without clipping 
# np.clip forces any number below 0.1 to become 0.1, and any above 2 to become 2.
raw_scores = np.random.normal(loc=0.5, scale=0.3, size=NUM_CITIZENS)
L_civics = np.clip(raw_scores, a_min=0.1, a_max=2.0)


# creation of data frame
df_citizens = pd.DataFrame({
    "citizen_id": citizen_ids,
    "join_date": join_dates,
    "L_civic": L_civics,
})

print(df_citizens.head())
print("\nL_civic:")
print(df_citizens["L_civic"].describe())

                             citizen_id   join_date   L_civic
0  f6e94851-7508-4eeb-8a91-bdae571e5b38  2026-04-20  0.798459
1  c92dc503-5abf-4d7b-b4ca-1fa870265289  2025-08-17  0.471951
2  4d1cb588-1f91-44b7-ba8c-876f8cdd0457  2025-11-03  1.152113
3  dbbc33db-dc12-477e-bd0d-67dcb5157246  2026-04-16  0.100000
4  ce7a943f-af41-4815-a3bc-42b668e02ad0  2026-05-21  0.211590

L_civic:
count    10000.000000
mean         0.511578
std          0.275631
min          0.100000
25%          0.296869
50%          0.499840
75%          0.701194
max          1.525584
Name: L_civic, dtype: float64


### Step 3B: Timeline & Latent Weather Shock
Simulate a 30-day calendar starting from  (August 1, 2026) and inject a hidden omitted variable  representing a decaying typhoon shock.

In [4]:
# Step 3B: Timeline & Latent Weather Shock

# 1. 30 consecutive days starting from BASE_DATE using timedelta list comprehension
# create a list of all dates from the starting base date
timeline_dates = [
    (BASE_DATE + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(30)
]

# 2. Vectorized initialization of latent storm shock (L_storm)
# intialize a 30 row list with 0.0
L_storm = np.zeros(30)

# 3. Inject decaying typhoon shock at specific indexes (Day 15, Day 16, Day 17)
# change the values to simulate a typoon situation
L_storm[14] = 1.0  # Index 14 (Day 15): Peak typhoon shock
L_storm[15] = 0.6  # Index 15 (Day 16): Receding floodwaters
L_storm[16] = 0.2  # Index 16 (Day 17): Residual shock

# Create timeline DataFrame
df_timeline = pd.DataFrame({
    "date": timeline_dates,
    "L_storm": L_storm
})

# Verify storm pulse injection
print("Timeline created. Storm pulse slice (index 13:18):")
print(df_timeline.iloc[13:18])

Timeline created. Storm pulse slice (index 13:18):
          date  L_storm
13  2026-08-14      0.0
14  2026-08-15      1.0
15  2026-08-16      0.6
16  2026-08-17      0.2
17  2026-08-18      0.0


### Step 3C: Timeline & Department Vulnerability Interaction Grid
Perform a Cartesian cross-join between `df_timeline` (30 days) and `df_departments` (5 departments) to create a 150-row simulation grid (`df_grid`).

In [ ]:
# Step 3C: Timeline & Department Vulnerability Interaction Grid

# Perform Cartesian cross-join between timeline and departments
# merge the timeline with departments table, creating 150 rows
df_grid = df_timeline.merge(df_departments, how="cross")

# Verification
print(f"Simulation grid created with shape: {df_grid.shape}")
print("\nSample rows during storm peak (2026-08-15):")
print(df_grid[df_grid["date"] == "2026-08-15"][["date", "name", "L_storm", "vulnerability_to_storm", "base_rate"]])


Simulation grid created with shape: (150, 7)

Sample rows during storm peak (2026-08-15):
          date                 name  L_storm  vulnerability_to_storm  base_rate
70  2026-08-15  Traffic & Transport      1.0                     5.0         20
71  2026-08-15         Public Works      1.0                     8.0         15
72  2026-08-15   Parks & Recreation      1.0                     1.5          2
73  2026-08-15       Animal Control      1.0                     1.0          5
74  2026-08-15           Sanitation      1.0                     3.0         45
